# ADA Pretraining 1 — Live FineWeb-Edu Pretraining

This notebook runs a bounded, real-data causal language-model pretraining session on a free Google Colab GPU. It streams FineWeb-Edu, trains the tokenizer on training documents, evaluates on a held-out stream, and saves resumable checkpoints and live metrics to Google Drive.

**Run order:** enable GPU → clone/install ADA → mount Drive → inspect architecture and real records → train → inspect live loss and saved checkpoints.

## What is being tested

ADA alternates an Mamba-2-style state-space reference mixer with causal low-rank attention. Every decoder block also combines sparse top-1 mixture-of-experts routing with differentiable Hopfield Energy Memory. This notebook pretrains that stack from random initialization using next-token cross-entropy plus an explicit MoE load-balancing term.

## Efficiency and power — precise interpretation

- **Efficiency hypotheses:** state-space sequence mixing, sparse top-1 expert dispatch, tied embeddings, and bounded sequence lengths may reduce compute or memory use; this notebook reports throughput and peak GPU memory for the selected configuration.
- **Current implementation boundary:** Mamba-2 uses a correctness-oriented PyTorch reference SSD implementation, not a fused CUDA/Triton kernel. This pilot does not establish comparative efficiency.
- **Interpretation:** falling training/validation loss shows optimization on this English educational corpus. It does not establish broad language ability, reasoning, safety, or production performance.


In [ ]:
# 1. Colab runtime check — choose Runtime > Change runtime type > T4 GPU (or better).
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Colab: Runtime > Change runtime type > GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
# 2. Clone the current ADA main branch and install pretraining dependencies.
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path('/content/ADA')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/vimalspaceton618-afk/ADA.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[pretraining]'], cwd=REPO_DIR, check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('ADA source:', REPO_DIR)
print('Commit:', commit)


In [ ]:
# 3. Mount Drive so the report and checkpoint survive a Colab disconnect.
from google.colab import drive

drive.mount('/content/drive')
RUN_DIR = pathlib.Path('/content/drive/MyDrive/ADA_Pretraining_1/fineweb_edu_live')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent artifact directory:', RUN_DIR)


## Architecture card

The bounded model uses four layers at width 128. Layer 0/2 use `Mamba2ReferenceMixer`; layer 1/3 use `TensorNetAttention`. Each layer includes top-1 MoE routing across four experts and a 64-slot Energy Memory. The tokenizer is learned from training split documents only.

This is a small pilot, not an architecture comparison. Record GPU, commit, tokens/sec, peak VRAM, and held-out validation loss to make later runs comparable.


In [ ]:
# 4. Inspect ADA's model configuration before pretraining.
from ada.neural import ADAConfig, ADAForCausalLM

config = ADAConfig(
    vocab_size=2048, dim=128, n_layers=4, n_heads=4, attention_rank=32,
    max_seq_len=128, num_experts=4, num_slots=64, expansion_factor=2,
    mixer_expand=2, mixer_kernel_size=4, temperature=0.1, dropout=0.0,
    tie_embeddings=True, mixer_backend='mamba2_reference',
    mamba2_d_state=16, mamba2_headdim=64, mamba2_ngroups=1, mamba2_chunk_size=32,
)
model = ADAForCausalLM(config)
print('Configuration:', config.to_dict())
print('Parameters:', f'{model.parameter_count():,}')
print('Mixer layout:', [type(layer.mixer).__name__ for layer in model.layers])
del model


## Preview the real corpus and start pretraining

The script streams pinned FineWeb-Edu records instead of downloading the full corpus, and creates a deterministic held-out validation split. The dataset card declares ODC-By 1.0 and notes Common Crawl terms also apply; review those terms before redistribution.


In [ ]:
# 5. Preview FineWeb-Edu, then configure live training.
from datasets import load_dataset
from pprint import pprint
preview = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True, revision='87f09149ef4734204d70ed1d046ddc9ca3f2b8f9')
for i, row in enumerate(preview.take(2), start=1):
    print(f'--- RECORD {i} ---')
    pprint({k: v for k, v in row.items() if k != 'text'})
    print((row.get('text') or '')[:800], '\n')

MAX_STEPS = 100
BATCH_SIZE = 2
SEQUENCE_LENGTH = 128
VOCAB_SIZE = 4096
TOKENIZER_DOCS = 2000
RESUME = False  # Set True only to resume this run with unchanged settings.

command = [
    sys.executable, 'scripts/ada_pretrain_fineweb_edu.py',
    '--device', 'cuda', '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE), '--sequence-length', str(SEQUENCE_LENGTH),
    '--vocab-size', str(VOCAB_SIZE), '--tokenizer-docs', str(TOKENIZER_DOCS),
    '--eval-every', '20', '--eval-batches', '4', '--save-every', '20',
    '--output-dir', str(RUN_DIR),
]
if RESUME:
    command.append('--resume')
print('Command:', ' '.join(command))
import time
training_started = time.time()
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
# 6. Plot live training and held-out validation metrics.
import json

import pandas as pd
import matplotlib.pyplot as plt
metrics_path = RUN_DIR / 'metrics.jsonl'
rows = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
metrics = pd.DataFrame(rows)
display(metrics.tail(10))
plt.figure(figsize=(10, 5))
for column in ('train_loss', 'validation_loss'):
    if column in metrics:
        points = metrics.dropna(subset=[column])
        if len(points): plt.plot(points['step'], points[column], marker='o', label=column)
plt.xlabel('Optimizer step'); plt.ylabel('Next-token cross-entropy'); plt.title('ADA live FineWeb-Edu pretraining')
plt.grid(True, alpha=0.25); plt.legend(); plt.show()


In [ ]:
# 7. Verify run metadata, checkpoint artifacts, and approximate throughput.
manifest_path = RUN_DIR / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('Run manifest:', json.dumps(manifest, indent=2))
else:
    print('No manifest yet; training may have stopped before the first checkpoint.')
for path in [RUN_DIR / 'tokenizer.json', RUN_DIR / 'model' / 'config.json', RUN_DIR / 'model' / 'model.pt', RUN_DIR / 'training_state.pt', metrics_path]:
    print(('OK  ' if path.exists() else 'MISS'), path, (f'{path.stat().st_size:,} bytes' if path.exists() and path.is_file() else ''))
elapsed = time.time() - training_started
tokens = MAX_STEPS * BATCH_SIZE * SEQUENCE_LENGTH
print(f'Elapsed wall time: {elapsed / 60:.1f} minutes')
print(f'Approximate training tokens processed: {tokens:,}')
print(f'Approximate end-to-end throughput: {tokens / max(elapsed, 1):.1f} tokens/s (includes data and evaluation overhead)')


## Decision after this notebook

This notebook runs the real FineWeb-Edu pilot directly. Keep the commit hash, run manifest, validation curve, and Drive checkpoint. Compare future changes only under the same GPU, steps, batch size, sequence length, precision, and data revision.

A falling loss indicates optimization on this dataset only. The next evidence gate is a documented held-out evaluation protocol and task-specific quality measurements.
